# PyTorch-Compatible Streaming Dataset Examples

This notebook demonstrates how to use the new streaming dataset classes for memory-efficient ML workflows.

## Features Demonstrated

1. **Streaming for Large Datasets** - Memory-efficient iteration
2. **Indexed Dataset with Shuffling** - Random access with caching
3. **Simple Batch Iterator** - Non-PyTorch batch processing
4. **Custom Transforms** - Data preprocessing and encoding
5. **Performance Comparison** - Memory usage and speed benchmarks
6. **Train/Test Split** - Creating separate datasets with WHERE clauses

In [2]:
# Import required libraries
import sys
import os
from pathlib import Path

# Add src to path if needed
project_root = Path.cwd().parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

import pbi
import pandas as pd
import numpy as np
from pbi import SequenceRetriever, PhageHostStreamingDataset, PhageHostIndexedDataset

# Try to import PyTorch (optional)
try:
    import torch
    from torch.utils.data import DataLoader
    TORCH_AVAILABLE = True
    print("✅ PyTorch available")
except ImportError:
    TORCH_AVAILABLE = False
    print("⚠️  PyTorch not available - some examples will be skipped")

print(f"PBI version: {pbi.__version__}")

⚠️  PyTorch not available - some examples will be skipped
PBI version: 0.1.0


In [3]:
# Connect to database using quick_connect
try:
    retriever = pbi.quick_connect()
    print("✅ Connected to database")
    
    # Get basic stats
    stats = retriever.get_stats()
    print("\n📊 Database Statistics:")
    print(f"  Phages: {stats['database']['phages']:,}")
    if 'hosts' in stats['database']:
        print(f"  Hosts: {stats['database']['hosts']:,}")
        print(f"  Phage-Host Associations: {stats['database'].get('phage_host_associations', 0):,}")
except Exception as e:
    print(f"❌ Error connecting: {e}")
    print("\n⚠️  Make sure you have run the workflow to create the database and FASTA files")
    raise

2026-02-05 14:23:26,700 - INFO - 📂 Checking FASTA index files:
2026-02-05 14:23:26,703 - INFO -    Phage index: True (52570.4 KB)
2026-02-05 14:23:26,708 - INFO -    Protein index: True (1432185.2 KB)
2026-02-05 14:23:26,712 - INFO - 📂 Using host mapping file: /data/processed/sequences/host_fasta_mapping.json
2026-02-05 14:23:26,720 - INFO -    Loaded mapping for 5362 hosts
2026-02-05 14:23:26,721 - INFO - 📂 Connecting to database: /data/processed/databases/phage_database_optimized.duckdb
2026-02-05 14:23:26,776 - INFO - 🔄 Starting background FASTA loading...
2026-02-05 14:23:26,777 - INFO - 🔄 [Background] Loading phage FASTA: /data/processed/sequences/all_phages.fasta
2026-02-05 14:23:26,778 - INFO - ✅ Initialization complete (FASTA loading in background)
2026-02-05 14:23:26,780 - INFO - ⏳ Waiting for FASTA loading to complete...


✅ Connected to database


2026-02-05 14:23:43,041 - INFO -    ✅ Phage FASTA loaded in 16.26s (873,717 sequences)
2026-02-05 14:23:43,043 - INFO - 🔄 [Background] Loading protein FASTA: /data/processed/sequences/all_proteins.fasta
2026-02-05 14:28:26,786 - WARNING - ⚠️ Timeout after 300s - FASTA may still be loading
2026-02-05 14:30:05,826 - INFO -    ✅ Protein FASTA loaded in 382.78s (31,050,116 sequences)
2026-02-05 14:30:05,829 - INFO -    ℹ️  Using on-demand loading for 5,362 individual host files
2026-02-05 14:30:05,830 - INFO - 🎉 All FASTA files loaded in 399.05s
2026-02-05 14:30:14,025 - INFO - 🔍 Sample phage keys:
2026-02-05 14:30:14,026 - INFO -    - 'AE002163.1...'
2026-02-05 14:30:14,027 - INFO -    - 'AF009630.1...'
2026-02-05 14:30:14,028 - INFO -    - 'AF011378.1...'
2026-02-05 14:30:14,029 - INFO - 🔍 Sample protein keys:
2026-02-05 14:30:14,031 - INFO -    - 'AE002163.1 AAF39720.1...'
2026-02-05 14:30:14,032 - INFO -    - 'AE002163.1 AAF39721.1...'
2026-02-05 14:30:14,032 - INFO -    - 'AE002163.1 


📊 Database Statistics:
  Phages: 873,718
  Hosts: 6,031
  Phage-Host Associations: 764,172


## Example 1: Streaming Dataset for Large Datasets

Use `PhageHostStreamingDataset` when:
- Working with very large datasets
- Memory is limited
- You don't need random access or shuffling
- Sequential processing is sufficient

In [4]:
print("=" * 80)
print("Example 1: Streaming Dataset for Large Datasets")
print("=" * 80)

# Create streaming dataset with filtering
# This only loads metadata for the query - sequences are fetched on-demand
streaming_dataset = retriever.create_streaming_dataset(
    where_clause="p.Completeness = 'complete'",  # Filter for complete phages
    batch_size=1000  # Fetch 1000 records at a time from database
)

print("\n✅ Created streaming dataset")
print("   Database batch size: 1000 records")
print("   Filter: Complete phages only")

if TORCH_AVAILABLE:
    # Use with PyTorch DataLoader
    dataloader = DataLoader(
        streaming_dataset,
        batch_size=32,  # Model batch size
        num_workers=0   # Use 0 for single worker (required for IterableDataset in notebooks)
    )
    
    print("\n📦 Iterating through batches...")
    
    # Process first few batches as demonstration
    max_batches = 5
    for i, batch in enumerate(dataloader):
        if i >= max_batches:
            break
        
        # Batch is a list of dictionaries
        print(f"\n  Batch {i+1}:")
        print(f"    Size: {len(batch)} samples")
        print(f"    Phage IDs: {batch[0]['Phage_ID'][:3] if len(batch) > 0 else 'None'}")
        print(f"    Phage sequence length: {len(batch[0]['Phage_Sequence']) if len(batch) > 0 else 0}")
        print(f"    Host sequence length: {len(batch[0]['Host_Sequence']) if len(batch) > 0 else 0}")
        
        # Here you would typically:
        # - Encode sequences
        # - Convert to tensors
        # - Train your model
    
    print(f"\n✅ Processed {max_batches} batches successfully")
else:
    # Manual iteration without PyTorch
    print("\n📦 Iterating manually (without PyTorch)...")
    
    max_samples = 10
    for i, sample in enumerate(streaming_dataset):
        if i >= max_samples:
            break
        
        print(f"\n  Sample {i+1}:")
        print(f"    Phage ID: {sample['Phage_ID']}")
        print(f"    Host ID: {sample['Host_ID']}")
        print(f"    Phage length: {len(sample['Phage_Sequence'])}")
        print(f"    Host length: {len(sample['Host_Sequence'])}")
    
    print(f"\n✅ Processed {max_samples} samples successfully")

2026-02-05 14:30:14,214 - WARNING - PyTorch not available. Dataset will work but cannot be used with DataLoader.
2026-02-05 14:30:14,218 - INFO - Using host mapping with 5362 hosts


Example 1: Streaming Dataset for Large Datasets

✅ Created streaming dataset
   Database batch size: 1000 records
   Filter: Complete phages only

📦 Iterating manually (without PyTorch)...

✅ Processed 10 samples successfully


## Example 2: Indexed Dataset with Shuffling

Use `PhageHostIndexedDataset` when:
- Dataset fits in memory (metadata cached)
- You need shuffling for better training
- You want multi-worker data loading
- Random access is needed

In [5]:
print("=" * 80)
print("Example 2: Indexed Dataset with Shuffling")
print("=" * 80)

# Create indexed dataset
# Metadata is cached in memory, sequences fetched on-demand
indexed_dataset = retriever.create_indexed_dataset(
    where_clause="p.Completeness = 'complete' AND h.Assembly_Level = 'Complete Genome'"  # High-quality pairs
)

print(f"\n✅ Created indexed dataset")
print(f"   Total samples: {len(indexed_dataset):,}")
print("   Filter: Complete phages + Complete host genomes")

if TORCH_AVAILABLE:
    # Use with PyTorch DataLoader - supports shuffling!
    dataloader = DataLoader(
        indexed_dataset,
        batch_size=32,
        shuffle=True,  # Enable shuffling for better training
        num_workers=0  # Can use >0 in production (not in notebooks)
    )
    
    print("\n📦 Iterating with shuffling enabled...")
    
    # Process first few batches
    max_batches = 3
    for i, batch in enumerate(dataloader):
        if i >= max_batches:
            break
        
        print(f"\n  Batch {i+1}:")
        print(f"    Size: {len(batch['Phage_ID'])} samples")
        print(f"    Phage IDs: {batch['Phage_ID'][:3]}")
        print(f"    Host Species: {batch['Species_Name'][:3]}")
    
    print(f"\n✅ Shuffled iteration successful")
else:
    # Direct access by index (without PyTorch)
    print("\n📦 Random access examples (without PyTorch)...")
    
    if len(indexed_dataset) == 0:
        print("\n⚠️  Dataset is empty - no samples to show")
        print("   This may be due to data not being available or filters being too restrictive")
    else:
    
        # Access specific indices
        indices = [0, len(indexed_dataset)//2, len(indexed_dataset)-1]
        
        for idx in indices:
            sample = indexed_dataset[idx]
            print(f"\n  Sample at index {idx}:")
            print(f"    Phage ID: {sample['Phage_ID']}")
            print(f"    Host Species: {sample['Species_Name']}")
    
        print("\n✅ Random access successful")


2026-02-05 14:30:30,175 - WARNING - PyTorch not available. Dataset will work but cannot be used with DataLoader.
2026-02-05 14:30:30,179 - INFO - Using host mapping with 5362 hosts


Example 2: Indexed Dataset with Shuffling


2026-02-05 14:30:30,412 - INFO - Loaded metadata for 0 phage-host pairs



✅ Created indexed dataset
   Total samples: 0
   Filter: Complete phages + Complete host genomes

📦 Random access examples (without PyTorch)...


IndexError: list index out of range

## Example 3: Simple Batch Iterator

Use `get_phage_host_pairs_iterator()` when:
- You don't need PyTorch
- You want DataFrame batches
- Simple batch processing is sufficient
- Working with pandas-based workflows

In [7]:
print("=" * 80)
print("Example 3: Simple Batch Iterator (Non-PyTorch)")
print("=" * 80)

# Create iterator that yields DataFrame batches
batch_iterator = retriever.get_phage_host_pairs_iterator(
    where_clause="p.Length > 10000",  # Filter for larger phages
    batch_size=500
)

print("\n✅ Created batch iterator")
print("   Batch size: 500 pairs per batch")
print("   Filter: Phages > 10kb")

# Process batches
print("\n📦 Processing batches...")

max_batches = 3
all_gc_content = []

for i, batch_df in enumerate(batch_iterator):
    if i >= max_batches:
        break
    
    print(f"\n  Batch {i+1}:")
    print(f"    Rows: {len(batch_df)}")
    print(f"    Columns: {list(batch_df.columns)}")
    
    # Example analysis: compute GC content statistics
    avg_phage_gc = batch_df['Phage_GC'].mean()
    avg_host_gc = batch_df['Host_GC'].mean()
    
    print(f"    Avg Phage GC: {avg_phage_gc:.2f}%")
    print(f"    Avg Host GC: {avg_host_gc:.2f}%")
    
    all_gc_content.extend(batch_df['Phage_GC'].tolist())
    
    # You could export each batch to a file:
    # batch_df.to_csv(f'batch_{i}.csv', index=False)

print(f"\n✅ Processed {max_batches} batches")
print(f"   Total GC content values collected: {len(all_gc_content)}")

2026-02-16 11:36:26,713 - INFO - 🔍 Starting batch iteration with batch_size=500


Example 3: Simple Batch Iterator (Non-PyTorch)

✅ Created batch iterator
   Batch size: 500 pairs per batch
   Filter: Phages > 10kb

📦 Processing batches...


2026-02-16 11:36:31,770 - INFO - 📦 Processing batch 1 (984166 pairs)
2026-02-16 11:36:38,167 - WARNING - ⚠️  Phage sequence not found for ID: BK000583.1
2026-02-16 11:38:33,645 - WARNING - ⚠️  Error retrieving host sequence for GCF_977057215_1: Could not read index file /data/intermediate/fasta/hosts/GCF_977057215_1.fna.fai
2026-02-16 11:38:33,648 - WARNING - ⚠️  Error retrieving host sequence for GCF_030825475_1: Could not read index file /data/intermediate/fasta/hosts/GCF_030825475_1.fna.fai
2026-02-16 11:38:33,649 - WARNING - ⚠️  Error retrieving host sequence for GCF_009833085_1: Could not read index file /data/intermediate/fasta/hosts/GCF_009833085_1.fna.fai
2026-02-16 11:38:33,658 - WARNING - ⚠️  Error retrieving host sequence for GCF_040438935_1: Could not read index file /data/intermediate/fasta/hosts/GCF_040438935_1.fna.fai
2026-02-16 11:38:33,659 - WARNING - ⚠️  Error retrieving host sequence for GCF_023264895_1: Could not read index file /data/intermediate/fasta/hosts/GCF_02


  Batch 1:
    Rows: 984166
    Columns: ['Phage_ID', 'Host_ID', 'Phage_Source', 'Phage_Length', 'Phage_GC', 'Phage_Taxonomy', 'Phage_Completeness', 'Phage_Lifestyle', 'Phage_Cluster', 'Phage_Subcluster', 'Species_Name', 'Host_Assembly_Level', 'Host_Length', 'Host_GC', 'Host_RefSeq_Category', 'Phage_Sequence', 'Host_Sequence']
    Avg Phage GC: 44.75%
    Avg Host GC: 44.91%


2026-02-16 11:38:52,221 - INFO - 📦 Processing batch 2 (728346 pairs)
2026-02-16 11:40:14,222 - WARNING - ⚠️  Error retrieving host sequence for GCF_000321165_1: Could not read index file /data/intermediate/fasta/hosts/GCF_000321165_1.fna.fai
2026-02-16 11:40:14,829 - WARNING - ⚠️  Error retrieving host sequence for GCF_052366725_1: Could not read index file /data/intermediate/fasta/hosts/GCF_052366725_1.fna.fai
2026-02-16 11:40:14,853 - WARNING - ⚠️  Error retrieving host sequence for GCF_000176235_1: Could not read index file /data/intermediate/fasta/hosts/GCF_000176235_1.fna.fai
2026-02-16 11:40:14,893 - WARNING - ⚠️  Error retrieving host sequence for GCF_004345725_1: Could not read index file /data/intermediate/fasta/hosts/GCF_004345725_1.fna.fai
2026-02-16 11:40:14,894 - WARNING - ⚠️  Error retrieving host sequence for GCF_051376105_1: Could not read index file /data/intermediate/fasta/hosts/GCF_051376105_1.fna.fai
2026-02-16 11:40:14,914 - WARNING - ⚠️  Error retrieving host sequ


  Batch 2:
    Rows: 728346
    Columns: ['Phage_ID', 'Host_ID', 'Phage_Source', 'Phage_Length', 'Phage_GC', 'Phage_Taxonomy', 'Phage_Completeness', 'Phage_Lifestyle', 'Phage_Cluster', 'Phage_Subcluster', 'Species_Name', 'Host_Assembly_Level', 'Host_Length', 'Host_GC', 'Host_RefSeq_Category', 'Phage_Sequence', 'Host_Sequence']
    Avg Phage GC: 44.83%
    Avg Host GC: 44.65%

✅ Processed 3 batches
   Total GC content values collected: 1712512


## Example 4: Custom Transforms

Apply custom transformations to data:
- Sequence encoding (one-hot, k-mers)
- Feature extraction
- Data augmentation
- Normalization

In [8]:
print("=" * 80)
print("Example 4: Custom Transforms")
print("=" * 80)

# Define custom transform function
def sequence_encoding_transform(sample):
    """
    Transform that adds encoded features to the sample.
    
    This example computes:
    - Sequence length features
    - GC content features
    - Simple nucleotide counts
    """
    phage_seq = sample['Phage_Sequence']
    host_seq = sample['Host_Sequence']
    
    # Length features
    sample['phage_length'] = len(phage_seq)
    sample['host_length'] = len(host_seq)
    sample['length_ratio'] = len(phage_seq) / max(len(host_seq), 1)
    
    # Nucleotide composition for phage
    phage_upper = phage_seq.upper()
    sample['phage_gc_computed'] = ((phage_upper.count('G') + phage_upper.count('C')) / 
                                    max(len(phage_seq), 1)) * 100
    sample['phage_a_count'] = phage_upper.count('A')
    sample['phage_t_count'] = phage_upper.count('T')
    sample['phage_g_count'] = phage_upper.count('G')
    sample['phage_c_count'] = phage_upper.count('C')
    
    # Optionally keep or remove raw sequences to save memory
    # del sample['Phage_Sequence']
    # del sample['Host_Sequence']
    
    return sample

# Create dataset with transform
transformed_dataset = retriever.create_indexed_dataset(
    where_clause="p.Completeness = 'complete' LIMIT 100",
    transform=sequence_encoding_transform
)

print(f"\n✅ Created dataset with custom transform")
print(f"   Total samples: {len(transformed_dataset)}")

if len(transformed_dataset) == 0:
    print("\n⚠️  Dataset is empty - cannot show sample")
    print("   This may be due to data not being available or filters being too restrictive")
else:

    # Get a sample to see the transformed data
    sample = transformed_dataset[0]
    print("\n📊 Transformed sample fields:")
    for key in sorted(sample.keys()):
        if key not in ['Phage_Sequence', 'Host_Sequence']:  # Skip long sequences in print
            print(f"   {key}: {sample[key]}")

    print("\n✅ Transform applied successfully")

2026-02-16 11:40:38,691 - WARNING - PyTorch not available. Dataset will work but cannot be used with DataLoader.
2026-02-16 11:40:38,695 - INFO - Using host mapping with 5362 hosts


Example 4: Custom Transforms


2026-02-16 11:40:38,892 - INFO - Loaded metadata for 0 phage-host pairs



✅ Created dataset with custom transform
   Total samples: 0


IndexNotFoundError: Could not read index file /data/processed/sequences/all_phages.fasta.fai

## Example 5: Performance Comparison

Compare different approaches:
- Loading all at once vs streaming
- Memory usage
- Iteration speed

In [ ]:
print("=" * 80)
print("Example 5: Performance Comparison")
print("=" * 80)

import time
import gc

# Test with a limited dataset for comparison
test_where = "p.Completeness = 'complete' LIMIT 500"

print("\n🔬 Test Configuration:")
print(f"   Filter: {test_where}")

# Method 1: Load all at once (traditional approach)
print("\n📊 Method 1: Load All At Once")
gc.collect()
start_time = time.time()

all_at_once_df = retriever.get_phage_host_pairs(where_clause=test_where.replace(' LIMIT 500', ''))
all_at_once_df = all_at_once_df.head(500)  # Limit to 500 for fair comparison

load_all_time = time.time() - start_time
memory_mb = all_at_once_df.memory_usage(deep=True).sum() / 1024 / 1024

print(f"   Time: {load_all_time:.2f}s")
print(f"   DataFrame memory: {memory_mb:.2f} MB")
print(f"   Rows: {len(all_at_once_df)}")

# Method 2: Streaming dataset
print("\n📊 Method 2: Streaming Dataset")
gc.collect()
start_time = time.time()

streaming_ds = retriever.create_streaming_dataset(
    where_clause=test_where.replace(' LIMIT 500', ''),
    batch_size=100
)

count = 0
for i, sample in enumerate(streaming_ds):
    count += 1
    if count >= 500:
        break

streaming_time = time.time() - start_time

print(f"   Time: {streaming_time:.2f}s")
print(f"   Samples processed: {count}")
print(f"   Note: Streaming uses minimal memory (no full DataFrame)")

# Method 3: Batch iterator
print("\n📊 Method 3: Batch Iterator")
gc.collect()
start_time = time.time()

count = 0
for batch_df in retriever.get_phage_host_pairs_iterator(
    where_clause=test_where.replace(' LIMIT 500', ''),
    batch_size=100
):
    count += len(batch_df)
    if count >= 500:
        break

iterator_time = time.time() - start_time

print(f"   Time: {iterator_time:.2f}s")
print(f"   Samples processed: {count}")
print(f"   Note: Processes one batch at a time")

# Summary
print("\n📈 Performance Summary:")
print(f"   Load all at once: {load_all_time:.2f}s ({memory_mb:.2f} MB)")
print(f"   Streaming:        {streaming_time:.2f}s (minimal memory)")
print(f"   Batch iterator:   {iterator_time:.2f}s (batch-sized memory)")
print("\n💡 Recommendations:")
print("   - Use streaming for very large datasets (100k+ pairs)")
print("   - Use indexed dataset for medium datasets with shuffling")
print("   - Use batch iterator for pandas-based workflows")

## Example 6: Train/Test Split with Streaming

Create separate datasets for training and testing using WHERE clauses.
This avoids data leakage and allows efficient memory usage.

In [ ]:
print("=" * 80)
print("Example 6: Train/Test Split with Streaming")
print("=" * 80)

# Strategy: Split data using different quality thresholds
# Train on high-quality data, test on medium-quality data

# Training dataset: Complete phages with complete host genomes
train_dataset = retriever.create_streaming_dataset(
    where_clause="p.Completeness = 'complete' AND h.Assembly_Level = 'Complete Genome'",
    batch_size=1000
)

print("\n✅ Created training dataset")
print("   Filter: Complete phages + Complete host genomes")
print("   Type: Streaming (for large-scale training)")

# Test dataset: Complete phages with scaffold-level assemblies
test_dataset = retriever.create_indexed_dataset(
    where_clause="p.Completeness = 'complete' AND h.Assembly_Level = 'Scaffold'"
)

print("\n✅ Created test dataset")
print(f"   Filter: Complete phages + Scaffold assemblies")
print(f"   Size: {len(test_dataset)} samples")
print("   Type: Indexed (for easier evaluation)")

if TORCH_AVAILABLE:
    # Create DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        num_workers=0
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False,  # No shuffling for test set
        num_workers=0
    )
    
    print("\n📦 DataLoaders created")
    print("   Train: Streaming, no shuffle")
    print("   Test: Indexed, no shuffle")
    
    # Simulate training loop
    print("\n🔄 Simulating training loop...")
    
    # Process a few training batches
    print("\n  Training batches:")
    for i, batch in enumerate(train_loader):
        if i >= 3:
            break
        print(f"    Batch {i+1}: {len(batch)} samples")
        # Here: model.train(), forward pass, compute loss, backward pass, optimize
    
    # Process a few test batches
    print("\n  Test batches:")
    for i, batch in enumerate(test_loader):
        if i >= 3:
            break
        print(f"    Batch {i+1}: {len(batch['Phage_ID'])} samples")
        # Here: model.eval(), forward pass, compute metrics
    
    print("\n✅ Train/test split demonstration complete")
else:
    print("\n⚠️  PyTorch not available - skipping DataLoader demonstration")

print("\n💡 Alternative splitting strategies:")
print("   - By source: WHERE p.Source_DB = 'PhagesDB' (train) vs 'INPHARED' (test)")
print("   - By taxonomy: Different phage families for train/test")
print("   - Random split: Use modulo on Phage_ID hash for deterministic split")
print("   - Temporal: WHERE h.Download_Date < '2023-01-01' (train) vs >= (test)")

## Summary and Best Practices

### When to Use Each Approach

1. **PhageHostStreamingDataset**
   - ✅ Very large datasets (millions of pairs)
   - ✅ Limited memory available
   - ✅ Sequential processing is sufficient
   - ❌ No shuffling support
   - ❌ No random access

2. **PhageHostIndexedDataset**
   - ✅ Medium-sized datasets (thousands to hundreds of thousands)
   - ✅ Need shuffling for better training
   - ✅ Multi-worker data loading
   - ✅ Random access needed
   - ❌ Metadata must fit in memory

3. **get_phage_host_pairs_iterator()**
   - ✅ Pandas-based workflows
   - ✅ Don't need PyTorch
   - ✅ Batch processing
   - ✅ Simple integration
   - ❌ No PyTorch integration

### Performance Tips

- **Batch size**: Larger batches = fewer database queries, but more memory
- **WHERE clauses**: Filter early at the database level for efficiency
- **Transforms**: Apply heavy preprocessing in transforms, not in training loop
- **Workers**: Use num_workers > 0 in production (not in Jupyter notebooks)
- **Caching**: Indexed dataset caches metadata; streaming does not

### Data Quality

- Always filter for complete/high-quality genomes when possible
- Check for missing sequences (handled gracefully with warnings)
- Validate data before heavy computation
- Use WHERE clauses to ensure data quality

### Memory Management

- Streaming: O(batch_size) memory usage
- Indexed: O(metadata_size + batch_size) memory usage
- Load all: O(full_dataset) memory usage

Choose appropriately based on your dataset size and available RAM.

In [ ]:
# Cleanup
print("\n🧹 Cleaning up...")
retriever.close()
print("✅ Done!")